# IoT Traffic Analysis v9 — Packet-only (Time windows + MAC direction)

v9 introduces **time-based windows** (default 60s with 30s stride), improved long-gap representation, and **strict MAC-based direction**.

**Important:** The notebook will raise an error during setup if it cannot infer a confident device MAC (per your requirement).

In [ ]:
# If running in a fresh environment, install deps as needed:
# !pip install scapy tensorflow scikit-learn pandas numpy matplotlib

import os
import math
import random
import numpy as np
import pandas as pd

from scapy.all import PcapReader
from scapy.layers.l2 import Ether, Dot3
from scapy.layers.inet import IP, TCP, UDP, ICMP

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow:", tf.__version__)


## 1) Dataset location

In [ ]:
# === UPDATE THIS PATH ===
PCAP_DIR = "unsw-dataset/pcaps"   # e.g. "/content/pcaps" or "./pcaps"

assert os.path.isdir(PCAP_DIR), f"PCAP_DIR not found: {PCAP_DIR}"
pcap_files = sorted([os.path.join(PCAP_DIR, f) for f in os.listdir(PCAP_DIR) if f.endswith(".pcap")])

print("PCAP count:", len(pcap_files))
print("Example:", pcap_files[:3])


## 2) Mapping

In [ ]:
from collections import Counter

device_to_type_v7 = {
    "SamsungCamera_00166cab6b88": "camera_streamer",
    "BelkinCamera_b4750eece5a9": "camera_streamer",
    "NestDropCam_308cfb2fe4b2": "camera_streamer",
    "TPLinkCamera_f4f26D9351f1": "camera_streamer",
    "CanaryCamera_7c70bc5d5edc": "camera_streamer",
    "NetatmoWelcome_70ee50183443": "camera_streamer",
    "WithingsBabyMonitor_0024e41118a8": "camera_streamer",
    "RingDoorBell_884aea31669d": "camera_streamer",
    "AugustDoorBell_e076d03f00ae": "camera_streamer",

    "AwairAirQuality_70886b100fc6": "sensor_meter",
    "NetatmoWeatherStation_70ee5003b8ac": "sensor_meter",
    "WithingsSleepSensor_0024e42028c6": "sensor_meter",
    "WithingsSmartScale_0024e41b6f96": "sensor_meter",
    "BlipCareBPMeter_746a89002e25": "sensor_meter",
    "NestProtect_18b43025bee4": "sensor_meter",
    "BelkinWemoMotionSensor_ec1a59832811": "sensor_meter",

    "PhilipsHue_0017882b9a25": "actuator_light_plug",
    "LiFXBulb_d073d5018308": "actuator_light_plug",
    "TPLinkSmartPlug_50c7bf005639": "actuator_light_plug",
    "BelkinWemoSwitch_ec1a5979f489": "actuator_light_plug",

    "SamsungSmartThings_d052a800675e": "hub_gateway_bridge",
    "HPPrinter_705a0fe49bc0": "hub_gateway_bridge",

    "TribySpeaker_18B79E022044": "media_speaker_display",
    "PixStarPhotoFrame_e076d033bb85": "media_speaker_display",
    "HelloBarbie_28c2ddffa52d": "media_speaker_display",
    "iHome_74c63b29d71d": "media_speaker_display",
    "AmazonEcho_44650d56ccd3": "media_speaker_display",
}

v7_to_v8 = {
    "camera_streamer": "continuous_streaming",
    "sensor_meter": "low_rate_telemetry",
    "actuator_light_plug": "interactive_actuators",
    "media_speaker_display": "smart_media_clients",
    "hub_gateway_bridge": None,  # excluded
}

device_to_type = {dev: v7_to_v8.get(t, None) for dev, t in device_to_type_v7.items()}

DROP_NONE_TYPES = True
target_types = sorted([t for t in set(device_to_type.values()) if t is not None])
print("v9 target types:", target_types)
print("Counts:", Counter([t for t in device_to_type.values() if t is not None]))


## 3) Build device table

In [ ]:
def device_id_from_path(pcap_path: str) -> str:
    return os.path.basename(pcap_path).replace(".pcap", "")

rows = []
for p in pcap_files:
    dev = device_id_from_path(p)
    t = device_to_type.get(dev, None)
    if DROP_NONE_TYPES and t is None:
        continue
    if t is None:
        t = "unknown_infrastructure"
    rows.append({"device_id": dev, "device_type": t, "pcap_path": p})

df_dev = pd.DataFrame(rows).sort_values(["device_type", "device_id"]).reset_index(drop=True)
print("Devices included:", len(df_dev))
display(df_dev.head(10))
print(df_dev["device_type"].value_counts())


## 4) Train/test split by device

In [ ]:
types = sorted(df_dev["device_type"].unique())
type_to_idx = {t: i for i, t in enumerate(types)}
idx_to_type = {i: t for t, i in type_to_idx.items()}
df_dev["y"] = df_dev["device_type"].map(type_to_idx).astype(int)

train_df, test_df = train_test_split(
    df_dev,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=df_dev["device_type"]
)

print("Train devices:", len(train_df), "Test devices:", len(test_df))
print("Train distribution:\n", train_df["device_type"].value_counts())
print("Test distribution:\n", test_df["device_type"].value_counts())


## 5) Strict MAC inference (will error if not confident)

In [ ]:
import ipaddress
from collections import defaultdict

def is_private_ip(ip_str: str) -> bool:
    try:
        return ipaddress.ip_address(ip_str).is_private
    except Exception:
        return False

def infer_device_mac_strict(pcap_path: str, scan_packets: int = 4000, min_count: int = 50, dominance: float = 1.6):
    """Infer a single 'device' MAC from a per-device PCAP.
    Strict by design: raises RuntimeError if confidence is low.

    Heuristic:
    - Require Ethernet layer (MACs present).
    - Consider only packets with IP layer and private (RFC1918) source IP.
    - Count src MAC occurrences among those packets.
    - Choose top MAC if:
        * count >= min_count
        * top_count >= dominance * second_count  (or second_count==0)
    """
    mac_counts = defaultdict(int)
    seen = 0
    with PcapReader(pcap_path) as pcap:
        for pkt in pcap:
            # Some Ethernet captures decode as Ethernet II (Ether) while others decode as 802.3 (Dot3/LLC).
            # Both have src/dst MACs; treat either as valid L2.
            if pkt.haslayer(Ether):
                l2 = pkt[Ether]
            elif pkt.haslayer(Dot3):
                l2 = pkt[Dot3]
            else:
                raise RuntimeError(
                    f"PCAP frame has no Ether() or Dot3() layer (no MACs visible to Scapy). "
                    f"Cannot compute direction via MAC for: {pcap_path}"
                )

            if not pkt.haslayer(IP):
                continue
            ip = pkt[IP]
            if not is_private_ip(ip.src):
                continue
            mac = l2.src
            if mac is None or mac == "ff:ff:ff:ff:ff:ff":
                continue
            mac_counts[mac] += 1
            seen += 1
            if seen >= scan_packets:
                break

    if not mac_counts:
        raise RuntimeError(f"Could not infer device MAC: no private-src-IP packets seen in scan for: {pcap_path}")

    items = sorted(mac_counts.items(), key=lambda kv: kv[1], reverse=True)
    top_mac, top_cnt = items[0]
    second_cnt = items[1][1] if len(items) > 1 else 0

    if top_cnt < min_count:
        raise RuntimeError(f"Low confidence MAC inference (top_cnt<{min_count}). top={top_mac} cnt={top_cnt} file={pcap_path}")
    if second_cnt > 0 and top_cnt < dominance * second_cnt:
        raise RuntimeError(f"Ambiguous MAC inference (top not dominant). top={top_mac} {top_cnt}, second={second_cnt} file={pcap_path}")

    return top_mac, dict(items[:10])

# Quick sanity check on a few files
for p in pcap_files[:3]:
    dev = device_id_from_path(p)
    if device_to_type.get(dev, None) is None and DROP_NONE_TYPES:
        continue
    mac, top = infer_device_mac_strict(p)
    print(dev, "->", mac, "top_counts:", list(top.items())[:3])


## 6) v9 windowing + features

In [ ]:
# v9 windowing: time-based windows + fixed-length packet tensor via pad/trim
WIN_SECONDS = 60.0
STRIDE_SECONDS = 30.0
T_MAX = 256  # max packets per window for packet branch
MAX_WINDOWS_PER_DEVICE = 1200

# Packet feature shape
Fp = 19  # v9 adds 2 features: iat_big, iat_huge
# Summary feature shape
Fs = 18  # directionless totals + time/gap stats + protocol mix + endpoint diversity + (optional) asymmetry

MTU_NORM = 1500.0

def safe_payload_len(pkt) -> int:
    if pkt.haslayer(TCP):
        return int(len(bytes(pkt[TCP].payload)))
    if pkt.haslayer(UDP):
        return int(len(bytes(pkt[UDP].payload)))
    return 0

def tcp_flag_bits(tcp_layer) -> np.ndarray:
    # SYN, ACK, FIN, RST, PSH, URG
    def has(flag_char):
        return 1.0 if flag_char in str(tcp_layer.flags) else 0.0
    return np.array([has("S"), has("A"), has("F"), has("R"), has("P"), has("U")], dtype=np.float32)

def is_keepalive_like(pkt) -> float:
    if not pkt.haslayer(TCP):
        return 0.0
    pl = safe_payload_len(pkt)
    if pl > 1:
        return 0.0
    fl = str(pkt[TCP].flags)
    if ("A" in fl) and ("S" not in fl) and ("F" not in fl) and ("R" not in fl) and len(pkt) <= 80:
        return 1.0
    return 0.0

def log1p_clip(x: float, cap: float = 300.0) -> float:
    x = min(max(x, 0.0), cap)
    return math.log1p(x)

def pad_or_trim_pkt_feats(pkt_feats: list, t_max: int = T_MAX) -> np.ndarray:
    if len(pkt_feats) >= t_max:
        arr = np.stack(pkt_feats[:t_max], axis=0).astype(np.float32)
    else:
        arr = np.zeros((t_max, Fp), dtype=np.float32)
        if len(pkt_feats) > 0:
            arr[:len(pkt_feats), :] = np.stack(pkt_feats, axis=0).astype(np.float32)
    return arr

def summarize_window(items: list, duration_s: float, device_mac: str | None, no_direction: bool = False) -> np.ndarray:
    n = len(items)
    if n == 0:
        return np.zeros((Fs,), dtype=np.float32)

    total_bytes = sum(it["plen"] for it in items)
    total_pkts = n
    pps = total_pkts / max(duration_s, 1e-6)
    bps = total_bytes / max(duration_s, 1e-6)

    iats = [it["iat"] for it in items[1:]] if n > 1 else [0.0]
    iat_mean = float(np.mean(iats)) if iats else 0.0
    iat_p90 = float(np.percentile(iats, 90)) if len(iats) >= 2 else iat_mean
    iat_max = float(np.max(iats)) if iats else 0.0
    frac_gt_1 = sum(1 for x in iats if x > 1.0) / max(len(iats), 1)
    frac_gt_5 = sum(1 for x in iats if x > 5.0) / max(len(iats), 1)

    tcp = sum(1 for it in items if it["proto"] == "tcp")
    udp = sum(1 for it in items if it["proto"] == "udp")
    icmp = sum(1 for it in items if it["proto"] == "icmp")
    other = n - tcp - udp - icmp

    dns_like = sum(1 for it in items if it["dns_like"])
    tls_like = sum(1 for it in items if it["tls_like"])

    unique_dsts = len(set(it["dst"] for it in items if it["dst"] is not None))

    if no_direction or device_mac is None:
        asym_bytes = 0.0
    else:
        up_bytes = sum(it["plen"] for it in items if it["dir"] == 1.0)
        down_bytes = total_bytes - up_bytes
        asym_bytes = (up_bytes - down_bytes) / (total_bytes + 1e-6)

    feat = np.array([
        math.log1p(total_pkts),                 # 0
        math.log1p(total_bytes),                # 1
        math.log1p(duration_s),                 # 2
        math.log1p(pps),                        # 3
        math.log1p(bps),                        # 4
        log1p_clip(iat_mean),                   # 5
        log1p_clip(iat_p90),                    # 6
        log1p_clip(iat_max),                    # 7
        frac_gt_1,                              # 8
        frac_gt_5,                              # 9
        tcp / n,                                # 10
        udp / n,                                # 11
        icmp / n,                               # 12
        other / n,                              # 13
        dns_like / n,                           # 14
        tls_like / n,                           # 15
        math.log1p(unique_dsts),                # 16
        asym_bytes,                             # 17
    ], dtype=np.float32)

    assert feat.shape[0] == Fs
    return feat

def iter_time_windows(pcap_path: str, label: int, no_direction: bool = False, max_windows: int = MAX_WINDOWS_PER_DEVICE):
    device_mac, _top = infer_device_mac_strict(pcap_path)  # strict: raises if fails
    windows_yielded = 0

    cur_items = []
    cur_pkt_feats = []
    win_start = None
    last_time = None
    next_emit_time = None

    with PcapReader(pcap_path) as pcap:
        for pkt in pcap:
            if not pkt.haslayer(IP) or not pkt.haslayer(Ether):
                continue

            t = float(getattr(pkt, "time", 0.0))
            if win_start is None:
                win_start = t
                next_emit_time = win_start + WIN_SECONDS

            if last_time is None:
                iat = 0.0
            else:
                iat = max(0.0, t - last_time)
            last_time = t

            # Strict L2 handling: Scapy may decode Ethernet as Ether() (Ethernet II) or Dot3() (802.3/LLC).
            if pkt.haslayer(Ether):
                l2 = pkt[Ether]
            elif pkt.haslayer(Dot3):
                l2 = pkt[Dot3]
            else:
                raise RuntimeError("Frame has no Ether() or Dot3() layer (no MACs visible to Scapy).")

            ip = pkt[IP]
            plen = float(len(pkt))
            payload_len = float(safe_payload_len(pkt))
            ttl = float(getattr(ip, "ttl", 0.0))

            direction = 0.0
            if not no_direction:
                direction = 1.0 if l2.src == device_mac else 0.0

            proto = int(getattr(ip, "proto", 0))
            if proto == 6:
                proto_oh = np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float32)
                proto_name = "tcp"
            elif proto == 17:
                proto_oh = np.array([0.0, 1.0, 0.0, 0.0], dtype=np.float32)
                proto_name = "udp"
            elif proto == 1:
                proto_oh = np.array([0.0, 0.0, 1.0, 0.0], dtype=np.float32)
                proto_name = "icmp"
            else:
                proto_oh = np.array([0.0, 0.0, 0.0, 1.0], dtype=np.float32)
                proto_name = "other"

            flags = np.zeros(6, dtype=np.float32)
            syn = rst = False
            dns_like = tls_like = False

            if pkt.haslayer(TCP):
                tcp = pkt[TCP]
                flags = tcp_flag_bits(tcp)
                fl = str(tcp.flags)
                syn = ("S" in fl)
                rst = ("R" in fl)
                sport = int(getattr(tcp, "sport", 0))
                dport = int(getattr(tcp, "dport", 0))
                dns_like = (sport == 53 or dport == 53)
                tls_like = (sport == 443 or dport == 443)
            elif pkt.haslayer(UDP):
                udp = pkt[UDP]
                sport = int(getattr(udp, "sport", 0))
                dport = int(getattr(udp, "dport", 0))
                dns_like = (sport == 53 or dport == 53)
                tls_like = False

            keepalive = is_keepalive_like(pkt)

            pkt_len_norm = min(plen / MTU_NORM, 4.0)
            payload_norm = min(payload_len / MTU_NORM, 4.0)
            ttl_norm = ttl / 255.0
            iat_log = log1p_clip(iat, cap=300.0) / log1p_clip(300.0, cap=300.0)
            iat_big = 1.0 if iat > 1.0 else 0.0
            iat_huge = 1.0 if iat > 10.0 else 0.0

            pkt_feat = np.concatenate([
                np.array([pkt_len_norm, payload_norm, iat_log, direction, ttl_norm, iat_big, iat_huge], dtype=np.float32),
                proto_oh,
                flags,
                np.array([keepalive, 1.0 if dns_like else 0.0], dtype=np.float32),
            ], axis=0).astype(np.float32)

            assert pkt_feat.shape[0] == Fp

            cur_items.append({
                "plen": plen,
                "iat": iat,
                "proto": proto_name,
                "dst": getattr(ip, "dst", None),
                "dir": direction,
                "syn": syn,
                "rst": rst,
                "dns_like": dns_like,
                "tls_like": tls_like,
            })
            cur_pkt_feats.append(pkt_feat)

            if t >= next_emit_time:
                duration_s = max(t - win_start, 1e-6)
                X_pkt = pad_or_trim_pkt_feats(cur_pkt_feats, t_max=T_MAX)
                X_sum = summarize_window(cur_items, duration_s, device_mac=device_mac, no_direction=no_direction)
                yield X_pkt, X_sum, label

                windows_yielded += 1
                if windows_yielded >= max_windows:
                    break

                # Approximate stride by keeping the last half of samples (stride=WIN/2 default)
                keep_from = max(0, len(cur_items) // 2)
                cur_items = cur_items[keep_from:]
                cur_pkt_feats = cur_pkt_feats[keep_from:]
                win_start = win_start + STRIDE_SECONDS
                next_emit_time = win_start + WIN_SECONDS


## 7) Packet-only CNN

In [ ]:
def make_pkt_cnn(n_classes: int) -> tf.keras.Model:
    inp = layers.Input(shape=(T_MAX, Fp), name="pkt_features")
    x = layers.Conv1D(64, 5, padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool1D(2)(x)

    x = layers.Conv1D(128, 5, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool1D(2)(x)

    x = layers.Conv1D(256, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.25)(x)

    out = layers.Dense(n_classes, activation="softmax")(x)
    model = models.Model(inp, out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

n_classes = len(types)
model = make_pkt_cnn(n_classes)
model.summary()


## 8) Train

In [ ]:
NO_DIRECTION = False

def make_ds_packet(df_sub: pd.DataFrame, no_direction: bool = False, batch_size: int = 64, shuffle: bool = True):
    def gen():
        for _, row in df_sub.iterrows():
            y = int(row["y"])
            p = row["pcap_path"]
            for Xpkt, Xsum, y_ in iter_time_windows(p, label=y, no_direction=no_direction):
                yield Xpkt, y_
    out_sig = (
        tf.TensorSpec(shape=(T_MAX, Fp), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32)
    )
    ds = tf.data.Dataset.from_generator(gen, output_signature=out_sig)
    if shuffle:
        ds = ds.shuffle(2048, seed=RANDOM_STATE, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds_packet(train_df, no_direction=NO_DIRECTION, batch_size=64, shuffle=True)
test_ds  = make_ds_packet(test_df,  no_direction=NO_DIRECTION, batch_size=64, shuffle=False)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
]

history = model.fit(train_ds, validation_data=test_ds, epochs=30, callbacks=callbacks)

plt.figure()
plt.plot(history.history["accuracy"], label="train acc")
plt.plot(history.history["val_accuracy"], label="val acc")
plt.legend(); plt.title("Accuracy"); plt.show()


## 9) Evaluate

In [ ]:
def collect_preds_packet_only(df_sub: pd.DataFrame, model: tf.keras.Model, no_direction: bool = False, max_windows_per_device: int = 200):
    all_true = []
    all_pred = []
    per_device = []

    for _, row in df_sub.iterrows():
        dev = row["device_id"]
        y_true = int(row["y"])
        pcap_path = row["pcap_path"]

        probs = []
        wcount = 0
        for Xpkt, Xsum, y_ in iter_time_windows(pcap_path, label=y_true, no_direction=no_direction, max_windows=max_windows_per_device):
            p = model.predict(Xpkt[None, ...], verbose=0)[0]
            probs.append(p)
            y_hat = int(np.argmax(p))
            all_true.append(y_true)
            all_pred.append(y_hat)
            wcount += 1

        if probs:
            probs_mean = np.mean(np.stack(probs, axis=0), axis=0)
            y_dev = int(np.argmax(probs_mean))
        else:
            probs_mean = np.zeros((n_classes,), dtype=np.float32)
            y_dev = -1

        per_device.append((dev, y_true, probs_mean, y_dev, wcount))

    return np.array(all_true), np.array(all_pred), per_device

def show_reports(win_true, win_pred, dev_preds):
    print("Window-level classification report:")
    print(classification_report(win_true, win_pred, target_names=types, digits=4))

    cm = confusion_matrix(win_true, win_pred, labels=list(range(n_classes)))
    plt.figure(figsize=(7, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title("Window-level confusion matrix")
    plt.colorbar()
    plt.xticks(range(n_classes), types, rotation=45, ha="right")
    plt.yticks(range(n_classes), types)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

    dev_true = []
    dev_pred = []
    print("\nPer-device predictions:")
    for dev, y_true, probs_mean, y_dev, wcount in dev_preds:
        if y_dev >= 0:
            dev_true.append(y_true)
            dev_pred.append(y_dev)
        print(f"- {dev:35s} true={idx_to_type[y_true]:22s} pred={idx_to_type.get(y_dev,'(none)'):22s} windows={wcount}")

    dev_true = np.array(dev_true)
    dev_pred = np.array(dev_pred)

    print("\nDevice-level classification report (mean-prob aggregation):")
    print(classification_report(dev_true, dev_pred, target_names=types, digits=4))

    cm2 = confusion_matrix(dev_true, dev_pred, labels=list(range(n_classes)))
    plt.figure(figsize=(7, 6))
    plt.imshow(cm2, interpolation="nearest")
    plt.title("Device-level confusion matrix")
    plt.colorbar()
    plt.xticks(range(n_classes), types, rotation=45, ha="right")
    plt.yticks(range(n_classes), types)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

win_true, win_pred, dev_preds = collect_preds_packet_only(test_df, model, no_direction=NO_DIRECTION, max_windows_per_device=200)
show_reports(win_true, win_pred, dev_preds)